# Evaluating Natural Language Explanations of Food Recommendation System XFoodRec Across Several LLMs

Authors: Junxuan Ling, Tianyi Liu, Anqi Zhou

# 0. Document Instructions

## 0.1. Git Version Control

After making modifications, please use Colab's function of "Save in Github to keep changes" (Github authorization required) to commit the changes to the repository (Your own branch if needed).

**NOTE**: When A modifies this notebook, but have not commited to Github, if B opens this notebook, they will see the Github old version.

## 0.2. Handle persistent data storage

Mount Google Drive instead of relying on Colab's temporary run-time storage.

**IMPORTANT:** Although Google Colab allows collaboration of the notebook, data and folders are originally in Junxuan's drive, this could cause issues when refering to the mounted directory path. To allow the code to work for all of us, (who's not junxuan) please create a **short-cut** of Junxuan's shared folder in our own Google Drive, path `MyDrive/`.

In [1]:
# load from drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 0.3 Setup and data loading

In [2]:
%pip install -q spacy pyarrow
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import spacy

project_dir = Path("/content/drive/MyDrive/XFoodRec_Project_Data")
data_dir = project_dir / "raw"
processed_dir = project_dir / "processed"
persona_path = data_dir / "personas.json"
recommendation_path = data_dir / "recommendations.json"
recipe_path = data_dir / "recipes.parquet"
nutrition_fields_path = processed_dir / "nutrition_fields.json"

In [4]:
with persona_path.open("r", encoding="utf-8") as f:
    personas = json.load(f)
with recommendation_path.open("r", encoding="utf-8") as f:
    recommendations = json.load(f)

personas_df = pd.json_normalize(personas)
recipes = pd.read_parquet(recipe_path)
recipes["recipe_id"] = recipes["recipe_id"].astype(str)

explanation_lemmas_df = pd.read_parquet(processed_dir / "explanation_lemmas.parquet")
recipe_lemmas_df = pd.read_parquet(processed_dir / "recipe_lemmas.parquet")
persona_lemmas_df = pd.read_parquet(processed_dir / "persona_lemmas.parquet")
with nutrition_fields_path.open("r", encoding="utf-8") as f:
    nutrition_fields = json.load(f)

print("Explanations:", len(explanation_lemmas_df))
print("Source recipes:", len(recipes))
print("Personas:", len(personas_df))

Explanations: 536
Source recipes: 702
Personas: 40


# 2. Entity Extraction, Fact Verification & Hallucination Detection

Matching helpers reused from Step 1


In [5]:
import re

import spacy

# Persona fields: the taste profile and the dietary needs.
TASTE_FIELDS = ["liked_ingredient", "disliked_ingredient", "favorite_cuisine"]
HEALTH_FIELDS = ["dietary_goal", "dietary_restriction", "food_allergy", "health_condition"]
# Ingredients are matched by their core noun phrase; cuisines and health values are matched as written.
NOUN_PHRASE_FIELDS = ["liked_ingredient", "disliked_ingredient"]
# Allergens are foods ("egg", "milk"), so they are not general health labels.
NOT_A_LABEL_FIELDS = ["food_allergy"]

MIN_PREFIX_LENGTH = 4  # word forms sharing a beginning match, e.g. "sugar" / "sugary"

text_nlp = spacy.load("en_core_web_sm", disable=["ner"])
# spaCy adds every processed string to its vocabulary, so keep the model's own word list as loaded.
SPACY_MODEL_WORDS = set(text_nlp.vocab.strings)
_base_forms = {}


def to_list(value):
    if isinstance(value, np.ndarray):
        value = value.tolist()
    return [str(v).lower() for v in value] if isinstance(value, (list, tuple)) else []


def base_forms(tokens):
    """spaCy lemma of each word on its own, so 'oats'/'oat' and 'potatoes'/'potato' align."""
    new = sorted({t for t in tokens if t not in _base_forms})
    for word, doc in zip(new, text_nlp.pipe(new)):
        _base_forms[word] = doc[0].lemma_.lower() if len(doc) else word
    return [_base_forms[t] for t in tokens]


def term_matches(token, term):
    shorter, longer = sorted((token, term), key=len)
    return token == term or (len(shorter) >= MIN_PREFIX_LENGTH and longer.startswith(shorter))


def contains_sequence(tokens, sequence):
    n = len(sequence)
    return n > 0 and any(tokens[i:i + n] == sequence for i in range(len(tokens) - n + 1))


def item_key(lemmas):
    """Core noun phrase of a persona item: 'oats certify gf' -> 'oat', 'frozen vegetable' -> 'vegetable'."""
    doc = text_nlp(" ".join(lemmas))
    chunks = list(doc.noun_chunks)
    if not chunks:
        return base_forms(lemmas)
    root = chunks[0].root
    words = [t.text for t in chunks[0] if t.i == root.i or (t.dep_ == "compound" and t.head.i == root.i)]
    return base_forms(words)


def remove_title(tokens, title):
    """Drop every occurrence of the recipe title, then an opening run of title words or stop words."""
    kept, i, n = [], 0, len(title)
    while i < len(tokens):
        if n and tokens[i:i + n] == title:
            i += n
        else:
            kept.append(tokens[i])
            i += 1
    title_words, start = set(title), 0
    while start < min(len(kept), n + 2) and (kept[start] in title_words or kept[start] in text_nlp.Defaults.stop_words):
        start += 1
    return kept[start:]


def nutrient_terms_from_data(recipe_columns, nutrition_fields):
    """Nutrients recorded in the data: the recipe nutrient columns ('dietaryfiber_per_serving [g]'
    -> 'fiber') and nutrition_fields.json. Compound column names keep their longest ending that is
    an English word in spaCy's vocabulary."""
    terms = {str(f).lower() for f in nutrition_fields}
    for column in recipe_columns:
        name = re.sub(r"\[[^\]]*\]", "", str(column).lower()).strip()
        if "_per_" not in name:
            continue
        word = name.split("_per_")[0]
        endings = [word[i:] for i in range(len(word)) if word[i:] in SPACY_MODEL_WORDS]
        terms.add(max(endings, key=len) if endings else word)
    return sorted(set(base_forms(sorted(terms))))


def noun_phrase_core(chunk, is_name=False):
    """Main noun of a spaCy noun phrase with the words that name it: compounds and -ing words
    ('olive oil', 'baking mix'); describing words are dropped ('frozen vegetables' -> 'vegetable').
    For ingredient names (is_name=True) an -ing word right before the phrase also belongs to it."""
    root = chunk.root
    words = [t for t in chunk if t.i == root.i or (t.head.i == root.i and (t.dep_ == "compound" or t.tag_ == "VBG"))]
    before, after = chunk.start - 1, chunk.end
    if is_name and before >= 0 and chunk.doc[before].tag_ == "VBG" and chunk.doc[before] not in words:
        words = [chunk.doc[before]] + words
    if is_name and after < len(chunk.doc) and chunk.doc[after].tag_ == "VBG" and chunk.doc[after].head.i == root.i:
        words = words + [chunk.doc[after]]  # "salad dressing" when spaCy reads "dressing" as a verb
    terms = base_forms([t.text.lower() for t in words])
    return [] if all(t in text_nlp.Defaults.stop_words for t in terms) else terms


def clean_ingredient_line(line):
    """Drop notes in brackets, including a note that started on the previous line ('in this recipe)')."""
    line = str(line).lower()
    if ")" in line and ("(" not in line or line.index(")") < line.index("(")):
        line = line[line.index(")") + 1:]
    return re.sub(r"\([^)]*\)?", " ", line).strip()


def ingredient_line_names(doc):
    """Ingredient names in one parsed ingredient line: its main noun phrase and phrases joined to it by
    'and' / 'or'. A line that continues the previous one (starts with 'and' / 'or') gives nothing, and a
    'noun' that takes an object ('use salted butter') is read as a verb."""
    if len(doc) and doc[0].pos_ == "CCONJ":
        return []
    chunks = [c for c in doc.noun_chunks if not any(child.dep_ == "dobj" for child in c.root.children)]
    if not chunks:  # a bare name that spaCy does not parse as a noun phrase, e.g. "tahini"
        terms = base_forms([t.text.lower() for t in doc if not (t.is_punct or t.is_space or t.like_num)])
        return [terms] if terms and not all(t in text_nlp.Defaults.stop_words for t in terms) else []
    main = [c for c in chunks
            if c.root.dep_ == "ROOT" or (c.root.head.dep_ == "ROOT" and c.root.dep_ in ("dobj", "compound"))]
    main = main or chunks[:1]
    for c in chunks:
        if c not in main and c.root.dep_ == "conj" and any(c.root.head.i == m.root.i for m in main):
            main.append(c)
    return [name for name in (noun_phrase_core(c, is_name=True) for c in main) if name]


def build_matching_resources(persona_lemmas_df, recipe_lemmas_df, recipe_table, nutrition_fields):
    persona_values = {}
    for row in persona_lemmas_df.to_dict("records"):
        field = row["source_field"]
        if field not in TASTE_FIELDS + HEALTH_FIELDS:
            continue
        lemmas = to_list(row["persona_text_lemmas"])
        terms = item_key(lemmas) if field in NOUN_PHRASE_FIELDS else base_forms(lemmas)
        if terms:
            persona_values.setdefault(str(row["persona_id"]), []).append((field, terms))

    taste_values = {tuple(t) for values in persona_values.values() for f, t in values if f in TASTE_FIELDS}
    health_label_values = sorted({
        tuple(t) for values in persona_values.values() for f, t in values
        if f in HEALTH_FIELDS and f not in NOT_A_LABEL_FIELDS and tuple(t) not in taste_values
    })
    titles = {
        str(row["recipe_id"]): base_forms(to_list(row["recipe_text_lemmas"]))
        for row in recipe_lemmas_df.to_dict("records") if row["source_field"] == "recipe_title"
    }

    line_recipes, line_texts = [], []
    for recipe_id, lines in zip(recipe_table["recipe_id"].astype(str), recipe_table["ingredients_title"]):
        for line in to_list(lines):
            line_recipes.append(recipe_id)
            line_texts.append(clean_ingredient_line(line))
    ingredient_names = {}
    for recipe_id, doc in zip(line_recipes, text_nlp.pipe(line_texts, batch_size=256)):
        for name in ingredient_line_names(doc):
            ingredient_names.setdefault(recipe_id, set()).add(tuple(name))
    # Food words are names used as ingredients in the recipe data. Persona likes / dislikes are not added:
    # they also hold dish types and habits ("desserts", "complicated recipes").
    food_terms = set().union(*ingredient_names.values())
    return {
        "persona_values": persona_values,
        "health_label_values": [list(v) for v in health_label_values],
        "titles": titles,
        "nutrient_terms": nutrient_terms_from_data(list(recipe_table.columns), nutrition_fields),
        "ingredient_names": ingredient_names,
        "food_terms": food_terms,
    }


def explanation_words(row, resources):
    """Words of one explanation (a row of explanation_lemmas_df), in base form, title removed."""
    terms = base_forms(to_list(row["explanation_lemmas"]))
    return remove_title(terms, resources["titles"].get(str(row["recipe_id"]), []))


recipe_table = recipes
matching_resources = build_matching_resources(persona_lemmas_df, recipe_lemmas_df, recipe_table, nutrition_fields)
print("Nutrients recorded in the data:", matching_resources["nutrient_terms"])
print("Health label values (goals, diets, conditions):", len(matching_resources["health_label_values"]))
print("Personas with values:", len(matching_resources["persona_values"]))
print("Recipes with ingredient names:", len(matching_resources["ingredient_names"]),
      "| food words:", len(matching_resources["food_terms"]))

Nutrients recorded in the data: ['calorie', 'carb', 'carbohydrate', 'cholesterol', 'fat', 'fiber', 'protein', 'sodium', 'sugar']
Health label values (goals, diets, conditions): 60
Personas with values: 40
Recipes with ingredient names: 702 | food words: 705


## 2.1. Extract three types of claims


In [6]:
# Parse every explanation once; 2.1 a-c read their claims from these parses.

# Same ids as the processed lemma table: "<persona id>_rank_<rank>".
claim_texts = {
    f"{entry['persona']['id']}_rank_{rank:02d}": rec["explanation"]
    for entry in recommendations
    for rank, rec in enumerate(entry["recommendations"], start=1)
}
assert set(claim_texts) == set(explanation_lemmas_df["recommendation_id"]), "ids differ from the lemma table"

claim_index = explanation_lemmas_df[["recommendation_id", "persona_id", "recipe_id", "rank"]].copy()
claim_ids = claim_index["recommendation_id"].tolist()
claim_docs = dict(zip(claim_ids, text_nlp.pipe([claim_texts[i] for i in claim_ids], batch_size=64)))

# Where the recipe title appears in each explanation (character spans), so claims inside it are skipped.
title_by_recipe = dict(zip(recipe_table["recipe_id"].astype(str), recipe_table["title"].astype(str).str.lower()))
claim_title_spans = {}
for row in claim_index.itertuples():
    title = title_by_recipe.get(str(row.recipe_id), "")
    text = claim_texts[row.recommendation_id].lower()
    claim_title_spans[row.recommendation_id] = [(m.start(), m.end()) for m in re.finditer(re.escape(title), text)] if title else []


def outside_title(span, recommendation_id):
    return not any(span.start_char < end and span.end_char > start for start, end in claim_title_spans[recommendation_id])


def sentence_number(span):
    return next(k for k, sentence in enumerate(span.doc.sents) if sentence.start <= span.start < sentence.end)


def is_negated(token):
    """spaCy's negation mark ('not', 'never') on the word or the words it hangs from, up to its verb,
    or the word being the object of 'without' ('without added sugars'). A 'not' with words of its own
    ('not only ... but also') does not negate."""
    chain = [token]
    while chain[-1].pos_ not in ("VERB", "AUX") and chain[-1].head.i != chain[-1].i:
        chain.append(chain[-1].head)
    return (any(child.dep_ == "neg" and not list(child.children) for t in chain for child in t.children)
            or any(t.dep_ == "prep" and t.lower_ == "without" for t in chain))


print("Explanations parsed:", len(claim_docs))
print("Explanations whose text contains the recipe title:", sum(bool(s) for s in claim_title_spans.values()))

Explanations parsed: 536
Explanations whose text contains the recipe title: 503


### a. Ingredient mentions

In [7]:
food_terms = matching_resources["food_terms"]

ingredient_claim_rows = []
for recommendation_id, doc in claim_docs.items():
    for chunk in doc.noun_chunks:
        if not outside_title(chunk, recommendation_id):
            continue
        # In "a mix of flavors" or "a variety of vegetables" the thing named comes after "of".
        if any(child.dep_ == "prep" and child.lower_ == "of" for child in chunk.root.children):
            continue
        core = noun_phrase_core(chunk)
        root = base_forms([chunk.root.text.lower()])
        if tuple(core) in food_terms:
            term = core
        elif tuple(root) in food_terms and root[0] not in text_nlp.Defaults.stop_words:
            term = root
        else:
            continue
        ingredient_claim_rows.append({
            "recommendation_id": recommendation_id,
            "claim_type": "ingredient",
            "sentence": sentence_number(chunk),
            "text": chunk.text,
            "term": " ".join(term),
            "negated": is_negated(chunk.root),
            "start_char": chunk.start_char,
            "end_char": chunk.end_char,
        })
ingredient_claims_df = pd.DataFrame(ingredient_claim_rows)

mentions_per_explanation = (
    ingredient_claims_df.groupby("recommendation_id").size()
    .reindex(claim_index["recommendation_id"], fill_value=0)
)
print(f"Ingredient mentions: {len(ingredient_claims_df)} in "
      f"{(mentions_per_explanation > 0).sum()} of {len(mentions_per_explanation)} explanations "
      f"({100 * (mentions_per_explanation > 0).mean():.1f}%)")
display(mentions_per_explanation.value_counts().sort_index().rename_axis("mentions in one explanation")
        .rename("explanations").to_frame())

top_ingredients = (
    ingredient_claims_df.groupby("term")
    .agg(mentions=("recommendation_id", "size"), explanations=("recommendation_id", "nunique"))
    .sort_values("mentions", ascending=False)
)
display(top_ingredients.head(15))

Ingredient mentions: 504 in 333 of 536 explanations (62.1%)


,explanations
mentions in one explanation,
0,203
1,188
2,123
3,19
4,2
5,1


,mentions,explanations
term,,
chicken,96,94
vegetable,63,63
rice,46,45
bean,28,26
banana,19,19
egg,16,16
spinach,15,13
salmon,15,14
chicken breast,11,11


### b. Nutrient claims

Each claim keeps what the sentence says about it:

| Column | What it holds | Example |
|---|---|---|
| `number`, `unit` | An amount given for the nutrient | "33.1g of protein" → 33.1, g; "161 calories" → 161, calorie; "low in calories at 159.6" → 159.6 |
| `qualifier` | The describing word tied to the nutrient in the sentence | "high in protein" → high; "a good source of fiber" → good; "low-calorie" → low; "protein-packed" → packed |

A nutrient word with neither ("combines protein and fiber") is a plain mention. Nutrients joined by "and" share their wording: in "rich in protein and fiber" both are "rich". A number that the sentence structure does not tie to a nutrient belongs to the closest nutrient before it in the same sentence, if that nutrient has no number yet.

Nutrient words said about the user rather than the dish, i.e. in a phrase owned by "your" ("your low-carb diet", "your favorite proteins", "complements your protein-rich meals"), are not nutrient claims. When such a phrase names a user label ("low carb", "high protein") it is a label claim in c.

In [8]:
nutrient_terms = matching_resources["nutrient_terms"]


def nutrient_name(word):
    """The recorded nutrient a word names ('carbs' -> 'carb', 'sugary' -> 'sugar'), or None."""
    base = base_forms([word.lower()])[0]
    names = [n for n in nutrient_terms if term_matches(base, n)]
    return base if base in names else next(iter(names), None)


def about_user(token):
    """The word sits in a phrase owned by the user ('your low-carb diet', 'your favorite proteins'):
    a second-person possessive on it or on the words it hangs from, up to its verb."""
    while True:
        if any(child.dep_ == "poss" and "2" in child.morph.get("Person") for child in token.children):
            return True
        if token.head.i == token.i or token.pos_ in ("VERB", "AUX"):
            return False
        token = token.head


def is_describing(token):
    """An adjective or past participle: 'high', 'lower', 'rich', 'packed'."""
    return token.tag_ in ("JJ", "JJR", "JJS", "VBN")


def is_amount(token):
    return bool(re.fullmatch(r"\d[\d,]*(\.\d+)?", token.text))


def words_in_front(noun):
    """Describing words placed before a noun: 'low' in 'low-calorie', 'good' in 'a good source'."""
    return [c for c in noun.children if c.dep_ == "amod" and c.i < noun.i and is_describing(c)]


def describing_word_of(prep):
    """The describing word a phrase like 'in calories' belongs to ('high in protein', 'packed with protein'):
    the closest one before it, also when spaCy hangs the phrase on the verb ('is low | in calories')."""
    governor = prep.head
    candidates = [governor] if is_describing(governor) else []
    candidates += [c for c in governor.children if c.dep_ in ("acomp", "attr", "conj") and is_describing(c)]
    return sorted((c for c in candidates if c.i < prep.i), key=lambda c: c.i)[-1:]


def read_nutrient_claim(token):
    """Describing words and amount (number, unit) of one nutrient word, from the spaCy parse."""
    qualifier, number, unit = words_in_front(token), None, None
    amounts = [c for c in token.children if c.dep_ == "nummod" and c.i < token.i and is_amount(c)]
    if amounts:  # "161 calories", "25g protein"
        number = amounts[0]
        units = [c for c in token.children if c.dep_ == "compound" and number.i < c.i < token.i]
        unit = units[0] if units else token
    # A nutrient joined by "and" to another one shares its wording: "rich in protein and fiber".
    anchor = token
    while anchor.dep_ == "conj" and anchor.head.is_alpha and nutrient_name(anchor.head.text):
        anchor = anchor.head
    if anchor is not token and not qualifier:  # "lower calories and fat"
        qualifier = words_in_front(anchor)
    holder = anchor.head
    if anchor.dep_ == "pobj" and holder.dep_ == "prep":
        qualifier += describing_word_of(holder)
        noun = holder.head
        if holder.lower_ == "of" and noun.pos_ in ("NOUN", "PROPN"):  # "a good source of protein", "21.4g of protein"
            qualifier += [] if qualifier else words_in_front(noun)
            counts = [c for c in noun.children if c.dep_ == "nummod" and is_amount(c)]
            if counts and number is None and anchor is token:
                number, unit = counts[0], noun
    elif anchor.dep_ in ("compound", "amod", "nmod", "npadvmod") and holder.i != anchor.i:
        if is_describing(holder):  # "protein-rich", "protein-packed"
            qualifier.append(holder)
        elif not qualifier:  # "a high protein content", "a moderate calorie option"
            qualifier += [w for w in words_in_front(holder) if w.i < anchor.i]
    elif anchor.dep_ == "dobj":  # "keeping calories low"
        qualifier += [c for c in holder.children if c.dep_ == "oprd" and is_describing(c)]
    return qualifier, number, unit


nutrient_claim_rows = []
for recommendation_id, doc in claim_docs.items():
    claims = []
    for token in doc:
        nutrient = token.is_alpha and nutrient_name(token.text)
        if nutrient and outside_title(doc[token.i:token.i + 1], recommendation_id) and not about_user(token):
            qualifier, number, unit = read_nutrient_claim(token)
            claims.append({"token": token, "term": nutrient, "qualifier": qualifier, "number": number, "unit": unit})
    # A number the parse does not tie to a nutrient ("low in calories at 159.6", "protein (28.3g)")
    # belongs to the closest nutrient before it in the same sentence.
    used = {c["number"].i for c in claims if c["number"] is not None}
    for amount in doc:
        if not is_amount(amount) or amount.i in used or not outside_title(doc[amount.i:amount.i + 1], recommendation_id):
            continue
        before = [c for c in claims if c["token"].i < amount.i and c["token"].sent.start == amount.sent.start]
        if before and before[-1]["number"] is None:
            before[-1]["number"] = amount
            before[-1]["unit"] = amount.head if amount.dep_ == "nummod" else None
    for c in claims:
        words = [c["token"]] + c["qualifier"] + [w for w in (c["number"], c["unit"]) if w is not None]
        span = doc[min(w.i for w in words):max(w.i for w in words) + 1]
        nutrient_claim_rows.append({
            "recommendation_id": recommendation_id,
            "claim_type": "nutrient",
            "sentence": sentence_number(span),
            "text": span.text,
            "term": c["term"],
            "qualifier": " ".join(w.lower_ for w in sorted(set(c["qualifier"]), key=lambda w: w.i)) or None,
            "number": float(c["number"].text.replace(",", "")) if c["number"] is not None else None,
            "unit": base_forms([c["unit"].lower_.rstrip(".")])[0] if c["unit"] is not None else None,
            "negated": is_negated(c["token"]),
            "start_char": span.start_char,
            "end_char": span.end_char,
        })
nutrient_claims_df = pd.DataFrame(nutrient_claim_rows)

claims_per_explanation = (
    nutrient_claims_df.groupby("recommendation_id").size()
    .reindex(claim_index["recommendation_id"], fill_value=0)
)
print(f"Nutrient claims: {len(nutrient_claims_df)} in "
      f"{(claims_per_explanation > 0).sum()} of {len(claims_per_explanation)} explanations "
      f"({100 * (claims_per_explanation > 0).mean():.1f}%)")

nutrient_forms = pd.DataFrame({
    "claims": [
        nutrient_claims_df["number"].notna().sum(),
        (nutrient_claims_df["number"].isna() & nutrient_claims_df["qualifier"].notna()).sum(),
        (nutrient_claims_df["number"].isna() & nutrient_claims_df["qualifier"].isna()).sum(),
        nutrient_claims_df["negated"].sum(),
    ]
}, index=["with a number", "with a describing word only", "plain mention", "negated (any of the above)"])
display(nutrient_forms)

display(
    nutrient_claims_df.groupby("term")
    .agg(claims=("recommendation_id", "size"), explanations=("recommendation_id", "nunique"),
         with_number=("number", "count"), with_describing_word=("qualifier", "count"))
    .sort_values("claims", ascending=False)
)
display(nutrient_claims_df["qualifier"].value_counts().head(15).rename_axis("describing word").rename("claims").to_frame())

Nutrient claims: 795 in 494 of 536 explanations (92.2%)


,claims
with a number,159
with a describing word only,581
plain mention,55
negated (any of the above),8


,claims,explanations,with_number,with_describing_word
term,,,,
protein,303,291,64,232
calorie,294,274,86,218
carbohydrate,65,65,8,51
fat,45,45,0,43
fiber,43,43,0,35
carb,34,33,1,25
sodium,9,8,0,7
sugar,2,2,0,1


,claims
describing word,
low,130
good,119
rich,70
moderate,58
lower,58
higher,41
high,31
packed,20
healthy,15


### c. Diet / health-label claims

A label claim is a goal, diet or health condition from the user profiles (the health label values of the shared helpers) named in the explanation outside the recipe title, e.g. "your weight loss goal", "a vegetarian option", "your gluten-free needs". A longer form of a label word also counts ("ketogenic" for keto, "energy-boosting" for energy boost); a shorter one does not ("main" is not maintenance). When one label sits inside a longer one ("vegetarian" in "jain vegetarian"), only the longer one is kept.

A label that contains a nutrient word ("low carb", "high protein") counts here only when it is said about the user ("your low-carb diet"); said about the dish ("a high-protein dish") it is a nutrient claim in b. General wording that names no label ("fits your dietary restrictions", "free from allergens") is not a label claim.

In [9]:
label_values = matching_resources["health_label_values"]
# A label with a nutrient word ("low carb") counts here only when said about the user ("your low-carb diet");
# said about the dish ("a low-carb dish") it is a nutrient claim in b.
nutrient_labels = {tuple(value) for value in label_values if any(nutrient_name(word) for word in value)}


def extends(word, part):
    """The explanation word is the label word or a longer form of it ('keto' -> 'ketogenic',
    'boost' -> 'booster'); a shorter word ('main' for 'maintenance') does not count."""
    return word == part or (len(part) >= MIN_PREFIX_LENGTH and word.startswith(part))


label_claim_rows = []
for recommendation_id, doc in claim_docs.items():
    words = [t for t in doc if not (t.is_punct or t.is_space)]  # "gluten-free" reads as "gluten free"
    bases = base_forms([t.lower_ for t in words])
    found = []
    for value in label_values:
        n = len(value)
        for i in range(len(words) - n + 1):
            if all(extends(base, part) for base, part in zip(bases[i:i + n], value)):
                found.append((doc[words[i].i:words[i + n - 1].i + 1], value))
    for span, value in found:
        # A value inside a longer one ("vegetarian" in "jain vegetarian") is not a claim of its own.
        if any(other.start <= span.start and span.end <= other.end and len(other) > len(span) for other, _ in found):
            continue
        if not outside_title(span, recommendation_id):
            continue
        if tuple(value) in nutrient_labels and not about_user(span.root):
            continue
        label_claim_rows.append({
            "recommendation_id": recommendation_id,
            "claim_type": "label",
            "sentence": sentence_number(span),
            "text": span.text,
            "term": " ".join(value),
            "negated": is_negated(span.root),
            "start_char": span.start_char,
            "end_char": span.end_char,
        })
label_claims_df = pd.DataFrame(label_claim_rows)

labels_per_explanation = (
    label_claims_df.groupby("recommendation_id").size()
    .reindex(claim_index["recommendation_id"], fill_value=0)
)
print(f"Diet / health-label claims: {len(label_claims_df)} in "
      f"{(labels_per_explanation > 0).sum()} of {len(labels_per_explanation)} explanations "
      f"({100 * (labels_per_explanation > 0).mean():.1f}%), negated: {label_claims_df['negated'].sum()}")
display(
    label_claims_df.groupby("term")
    .agg(claims=("recommendation_id", "size"), explanations=("recommendation_id", "nunique"))
    .sort_values("claims", ascending=False)
)

Diet / health-label claims: 252 in 231 of 536 explanations (43.1%), negated: 2


,claims,explanations
term,,
weight loss,52,52
muscle gain,45,45
vegetarian,35,35
energy boost,28,28
maintenance,25,25
medical management,13,13
vegan,11,11
high protein,8,8
anti inflammatory,7,7


### All claims

The three tables are stacked into `claims_df`, one row per claim, with the persona, recipe and rank of the explanation. Columns that do not apply to a claim type (`qualifier`, `number`, `unit` outside b) are empty.

In [10]:
claim_tables = {"ingredient": ingredient_claims_df, "nutrient": nutrient_claims_df, "label": label_claims_df}
claims_df = (
    pd.concat(claim_tables.values(), ignore_index=True)
    .merge(claim_index, on="recommendation_id", how="left", validate="many_to_one")
)
claims_df = claims_df[
    ["recommendation_id", "persona_id", "recipe_id", "rank", "claim_type", "sentence", "text", "term",
     "qualifier", "number", "unit", "negated", "start_char", "end_char"]
]

explanation_count = len(claim_index)
claim_summary = pd.DataFrame({
    "claims": [len(t) for t in claim_tables.values()] + [len(claims_df)],
    "negated": [int(t["negated"].sum()) for t in claim_tables.values()] + [int(claims_df["negated"].sum())],
    "explanations": [t["recommendation_id"].nunique() for t in claim_tables.values()]
                    + [claims_df["recommendation_id"].nunique()],
}, index=list(claim_tables) + ["any"])
claim_summary["percent of explanations"] = (100 * claim_summary["explanations"] / explanation_count).round(1)
display(claim_summary)
print("Explanations without any claim:", explanation_count - claims_df["recommendation_id"].nunique())

,claims,negated,explanations,percent of explanations
ingredient,504,1,333,62.1
nutrient,795,8,494,92.2
label,252,2,231,43.1
any,1551,11,527,98.3


Explanations without any claim: 9


## 2.2 Compare and verify extracted claims with the actual data

### a. Ingredients → recipe's ingredient list

### b. Nutrient claims → recipe's nutrition fields

### c. Diet/health-label claims → both the recipe's ingredients and the persona's restrictions/allergies/conditions